[Reference](https://pub.towardsai.net/openai-function-calling-works-great-until-you-have-340-tools-12-tenants-real-production-traffic-fe02da116e39)

In [1]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_contract_status",
            "description": "Retrieve the current status of a contract by ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "contract_id": {
                        "type": "string",
                        "description": "The unique contract identifier"
                    },
                    "include_history": {
                        "type": "boolean",
                        "description": "Whether to include revision history"
                    }
                },
                "required": ["contract_id"]
            }
        }
    }
]

## Layer 1:- The Tool Registry: The Heart of the Whole Thing

In [3]:
class ToolRegistryService:
    def __init__(self, db: AsyncSession, cache: Redis):
        self.db = db
        self.cache = cache

    async def get_tools_for_context(
        self,
        tenant_id: str,
        user_intent: str,
        max_tools: int = 15
    ) -> list[dict]:
        """
        Smart tool selection - don't just return all tools.
        Use embedding similarity to pick the most relevant ones.
        """
        cache_key = f"tools:{tenant_id}:{hash(user_intent)}"
        cached = await self.cache.get(cache_key)
        if cached:
            return json.loads(cached)

        # Get tenant's allowed tools
        allowed_tools = await self._get_tenant_tools(tenant_id)

        # If under threshold, just return them all
        if len(allowed_tools) <= max_tools:
            return [t.to_openai_schema() for t in allowed_tools]

        # Otherwise, semantic search to find relevant tools
        intent_embedding = await self._embed(user_intent)
        scored_tools = []

        for tool in allowed_tools:
            tool_embedding = await self._get_tool_embedding(tool.id)
            similarity = cosine_similarity(intent_embedding, tool_embedding)
            scored_tools.append((tool, similarity))

        # Sort by relevance, take top N
        scored_tools.sort(key=lambda x: x[1], reverse=True)
        top_tools = [t[0] for t in scored_tools[:max_tools]]

        result = [t.to_openai_schema() for t in top_tools]

        # Cache for 5 minutes
        await self.cache.setex(cache_key, 300, json.dumps(result))

        return result

    async def _get_tenant_tools(self, tenant_id: str) -> list[Tool]:
        """Tenant isolation - each BU only sees their tools"""
        return await self.db.execute(
            select(Tool)
            .join(TenantToolAccess)
            .where(
                TenantToolAccess.tenant_id == tenant_id,
                TenantToolAccess.is_active == True,
                Tool.is_active == True
            )
        ).scalars().all()

## Layer 2:-Schema Validation: Trust Nobody, Especially Not GPT

In [4]:
{
  "name": "get_contract_status",
  "arguments": {
    "contract_id": "CONTRACT_12345",
    "include_history": "yes"
  }
}

{'name': 'get_contract_status',
 'arguments': {'contract_id': 'CONTRACT_12345', 'include_history': 'yes'}}

In [5]:
class SchemaValidator:
    def validate_tool_call(
        self,
        tool_name: str,
        arguments: dict,
        tool_schema: dict
    ) -> tuple[bool, dict, list[str]]:
        """
        Returns: (is_valid, coerced_args, error_messages)
        We try to coerce before failing hard.
        """
        errors = []
        coerced = arguments.copy()

        properties = tool_schema.get("parameters", {}).get("properties", {})
        required = tool_schema.get("parameters", {}).get("required", [])

        # Check required fields
        for field in required:
            if field not in arguments:
                errors.append(f"Missing required field: {field}")

        # Type coercion and validation
        for field, schema in properties.items():
            if field not in arguments:
                continue

            value = arguments[field]
            expected_type = schema.get("type")

            # Try to coerce common mistakes
            if expected_type == "boolean" and isinstance(value, str):
                if value.lower() in ("yes", "true", "1"):
                    coerced[field] = True
                elif value.lower() in ("no", "false", "0"):
                    coerced[field] = False
                else:
                    errors.append(f"Cannot coerce '{value}' to boolean for field {field}")

            elif expected_type == "integer" and isinstance(value, str):
                try:
                    coerced[field] = int(value)
                except ValueError:
                    errors.append(f"Cannot coerce '{value}' to integer for field {field}")

            # Enum validation
            if "enum" in schema and coerced.get(field) not in schema["enum"]:
                errors.append(
                    f"Value '{value}' not in allowed values for {field}: {schema['enum']}"
                )

        is_valid = len(errors) == 0
        return is_valid, coerced, errors

## Layer 3:- The Execution Sandbox: No, You Cannot Call send_wire_transfer Without Auth

In [6]:
class FunctionExecutionSandbox:
    def __init__(self, permission_service, rate_limiter, audit_logger):
        self.permissions = permission_service
        self.rate_limiter = rate_limiter
        self.audit = audit_logger

    async def execute(
        self,
        function_name: str,
        arguments: dict,
        tenant_id: str,
        user_id: str,
        request_id: str
    ) -> FunctionResult:

        # Step 1: Permission check
        permitted = await self.permissions.can_execute(
            tenant_id=tenant_id,
            user_id=user_id,
            function_name=function_name,
            arguments=arguments  # Some functions check argument-level permissions
        )
        if not permitted:
            await self.audit.log_denied_call(
                function_name, tenant_id, user_id, request_id
            )
            return FunctionResult.permission_denied(function_name)

        # Step 2: Rate limiting
        rate_ok = await self.rate_limiter.check_and_increment(
            key=f"func:{tenant_id}:{function_name}",
            max_calls=50,
            window_seconds=60
        )
        if not rate_ok:
            return FunctionResult.rate_limited(function_name)

        # Step 3: Execute with timeout
        function = self._get_function(function_name)

        try:
            result = await asyncio.wait_for(
                function.execute(arguments),
                timeout=30.0
            )

            await self.audit.log_successful_call(
                function_name, arguments, result, tenant_id, user_id, request_id
            )

            return FunctionResult.success(result)

        except asyncio.TimeoutError:
            await self.audit.log_timeout(function_name, tenant_id, request_id)
            return FunctionResult.timeout(function_name)

        except Exception as e:
            await self.audit.log_error(function_name, str(e), tenant_id, request_id)
            return FunctionResult.error(function_name, str(e))

## Layer 4:- Retry Manager: Because Failures Are Just Opportunities to Spend More Money


In [7]:
class RetryManager:
    MAX_RETRIES = 3

    async def handle_failed_execution(
        self,
        error: FunctionError,
        call_history: list[dict],
        context: RequestContext
    ) -> RetryDecision:

        attempt_count = sum(
            1 for call in call_history
            if call["function"] == error.function_name
        )

        if attempt_count >= self.MAX_RETRIES:
            return RetryDecision.escalate_to_human(
                reason=f"Function {error.function_name} failed {attempt_count} times",
                last_error=error
            )

        # Generate helpful error message for the model
        human_readable_error = await self._format_error_for_model(error)

        return RetryDecision.retry(
            tool_result={
                "role": "tool",
                "tool_call_id": error.call_id,
                "content": json.dumps({
                    "error": True,
                    "message": human_readable_error,
                    "suggestion": error.suggestion  # "Try using contract_id format: CT-YYYY-NNNN"
                })
            }
        )

    async def _format_error_for_model(self, error: FunctionError) -> str:
        """Make errors actually useful for the model to learn from"""

        if error.type == ErrorType.NOT_FOUND:
            return (
                f"The requested resource was not found. "
                f"Error: {error.message}. "
                f"Please verify the identifier format and try again."
            )
        elif error.type == ErrorType.PERMISSION_DENIED:
            return (
                f"You don't have permission to access this resource. "
                f"Please let the user know they need elevated access."
            )
        elif error.type == ErrorType.VALIDATION:
            return (
                f"Invalid parameters: {error.message}. "
                f"Please correct the parameters and try again."
            )
        else:
            return f"Function execution failed: {error.message}"

## Layer 5:- Cost Tracking: The Part Every Startup Forgets Until Month 3

In [8]:
class CostTrackingMiddleware:
    async def track_request(
        self,
        request_id: str,
        tenant_id: str,
        user_id: str,
        model: str,
        messages: list,
        tool_calls: list,
        response: dict
    ):
        usage = response.get("usage", {})

        cost_usd = self._calculate_cost(
            model=model,
            input_tokens=usage.get("prompt_tokens", 0),
            output_tokens=usage.get("completion_tokens", 0)
        )

        await self.db.execute(
            insert(RequestCost).values(
                request_id=request_id,
                tenant_id=tenant_id,
                user_id=user_id,
                model=model,
                input_tokens=usage.get("prompt_tokens", 0),
                output_tokens=usage.get("completion_tokens", 0),
                tool_calls_count=len(tool_calls),
                cost_usd=cost_usd,
                timestamp=datetime.utcnow()
            )
        )

        # Check if tenant is approaching budget limit
        monthly_spend = await self._get_monthly_spend(tenant_id)
        budget_limit = await self._get_budget_limit(tenant_id)

        if monthly_spend > budget_limit * 0.9:  # 90% of budget
            await self.alert_service.send_budget_alert(
                tenant_id=tenant_id,
                current_spend=monthly_spend,
                budget_limit=budget_limit
            )

    def _calculate_cost(self, model: str, input_tokens: int, output_tokens: int) -> float:
        # GPT-4o pricing (update this when OpenAI changes their mind again)
        pricing = {
            "gpt-4o": {"input": 0.0000025, "output": 0.00001},
            "gpt-4o-mini": {"input": 0.00000015, "output": 0.0000006},
            "gpt-4-turbo": {"input": 0.00001, "output": 0.00003},
        }

        p = pricing.get(model, pricing["gpt-4o"])
        return (input_tokens * p["input"]) + (output_tokens * p["output"])